# Colorectal Cancer Tissue Classification — Exploration & Fine-Tuning Notebook

This notebook lets you:
1. **Test the existing model** `polejowska/resnet-50-finetuned-nct-crc-he-45k` on your own images
2. **Fine-tune your own model** on the CRC dataset from Hugging Face if you want a custom version

**9 Tissue Classes:** ADI (Adipose), BACK (Background), DEB (Debris), LYM (Lymphocytes), MUC (Mucus), MUS (Smooth Muscle), NORM (Normal Colon), STR (Stroma), TUM (Tumor)

**Runtime**: Enable GPU → Runtime > Change runtime type > T4 GPU

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers datasets evaluate accelerate Pillow

In [ ]:
# ── Cell 2: Test the existing model (no training needed) ─────────────────────
from transformers import pipeline
from PIL import Image

MODEL_ID = 'polejowska/resnet-50-finetuned-nct-crc-he-45k'
clf = pipeline('image-classification', model=MODEL_ID)

LABEL_DESCRIPTIONS = {
    'ADI': 'Adipose tissue',         'BACK': 'Background / glass',
    'DEB': 'Debris',                 'LYM': 'Lymphocytes',
    'MUC': 'Mucus',                  'MUS': 'Smooth muscle',
    'NORM': 'Normal colon mucosa',   'STR': 'Cancer-associated stroma',
    'TUM': 'Colorectal adenocarcinoma (Tumor)'
}

# To test with a local file: img = Image.open('/path/to/your/image.jpg')
# Upload a file in Colab:
from google.colab import files
print('Upload an H&E histopathological image patch to test:')
uploaded = files.upload()

for filename in uploaded:
    img = Image.open(filename)
    result = clf(img)
    print(f'\nResults for {filename}:')
    for r in result:
        desc = LABEL_DESCRIPTIONS.get(r['label'], r['label'])
        print(f"  {r['label']:6s} ({desc:35s})  {r['score']:.1%}")

In [ ]:
# ── Cell 3: Check model labels and config ─────────────────────────────────────
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(MODEL_ID)
print('Labels:      ', model.config.id2label)
print('Architecture:', model.config.model_type)
print('Num labels:  ', model.config.num_labels)

---
## Optional: Fine-tune your own model on the CRC dataset from Hugging Face

Run the cells below only if you want to train a custom model under your own HuggingFace account.  
Skip these if the existing `polejowska/resnet-50-finetuned-nct-crc-he-45k` model works for you.

**Dataset**: `owkin/nct-crc-he` — loaded directly from HF Hub (no Kaggle needed)  
- `nct_crc_he_1k` split — 999 training images  
- `crc_val_he_7k` split — 7,180 validation images (9 balanced classes)

In [ ]:
# ── Cell 4 (Optional): Load dataset from Hugging Face ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')  # saves checkpoints to Drive

from huggingface_hub import notebook_login
notebook_login()  # paste your HF token from huggingface.co/settings/tokens

from datasets import load_dataset
dataset = load_dataset('owkin/nct-crc-he')

train_ds = dataset['nct_crc_he_1k']   # 999 training images
eval_ds  = dataset['crc_val_he_7k']   # 7180 validation images
labels   = train_ds.features['label'].names

print('Classes:  ', labels)
print('Train size:', len(train_ds))
print('Eval size: ', len(eval_ds))

In [ ]:
# ── Cell 5 (Optional): Fine-tune EfficientNet-B0 ─────────────────────────────
import numpy as np
import evaluate
from torchvision.transforms import (
    RandomResizedCrop, RandomHorizontalFlip, Compose,
    Normalize, ToTensor, Resize, CenterCrop
)
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    TrainingArguments, Trainer, DefaultDataCollator
)

HF_USERNAME = 'your-hf-username'  # ← replace with your HuggingFace username
MODEL_REPO  = f'{HF_USERNAME}/colon-cancer-tissue'
checkpoint  = 'google/efficientnet-b0'

processor = AutoImageProcessor.from_pretrained(checkpoint)
model_ft  = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label={str(i): l for i, l in enumerate(labels)},
    label2id={l: str(i) for i, l in enumerate(labels)},
    ignore_mismatched_sizes=True
)

size = processor.size.get('shortest_edge', 224)
norm = Normalize(mean=processor.image_mean, std=processor.image_std)
train_tf = Compose([RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), norm])
val_tf   = Compose([Resize(size), CenterCrop(size), ToTensor(), norm])

def pre_train(b): b['pixel_values'] = [train_tf(i.convert('RGB')) for i in b['image']]; del b['image']; return b
def pre_val(b):   b['pixel_values'] = [val_tf(i.convert('RGB'))   for i in b['image']]; del b['image']; return b

accuracy = evaluate.load('accuracy')
def compute_metrics(ep): return accuracy.compute(predictions=np.argmax(ep[0], axis=1), references=ep[1])

trainer = Trainer(
    model=model_ft,
    args=TrainingArguments(
        output_dir='/content/drive/MyDrive/colon-cancer-model',
        remove_unused_columns=False, eval_strategy='epoch', save_strategy='epoch',
        learning_rate=5e-5, per_device_train_batch_size=16, num_train_epochs=5,
        load_best_model_at_end=True, metric_for_best_model='accuracy',
        push_to_hub=True, hub_model_id=MODEL_REPO,
    ),
    data_collator=DefaultDataCollator(),
    train_dataset=train_ds.with_transform(pre_train),
    eval_dataset=eval_ds.with_transform(pre_val),
    processing_class=processor,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.push_to_hub()
print(f'Done! Update MODEL_ID in app.py to: "{MODEL_REPO}"')